# 03h — Cross-validation k=5 sobre CatBoost tuneado

Re-evalúa el modelo con `best_params` de 03g sobre 5 folds estratificados
(train+val combinados). Test set se mantiene aparte.

In [1]:
# [SETUP]
import pandas as pd
import numpy as np
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score, log_loss, f1_score
import json
import time
from pathlib import Path
from datetime import datetime

RANDOM_STATE = 42
K = 5
TARGETS = ['churn_14d', 'churn_30d']

PROJECT_ROOT = Path.cwd().parents[1] if Path.cwd().name == 'fase3_modeling' else Path.cwd()
DATA_QC = PROJECT_ROOT / 'data' / 'data_qc'
DATA_MODELS = PROJECT_ROOT / 'data' / 'data_models'
REPORT_DIR = PROJECT_ROOT / 'informes' / 'fase3_modeling' / '03h_cv_catboost'
REPORT_DIR.mkdir(parents=True, exist_ok=True)

MASTER_PATH = DATA_QC / 'master_table_cutoff_v3_aggressive.parquet'
SPLITS_PATH = DATA_MODELS / 'splits_indices_cutoff.parquet'
TUNING_DIR = PROJECT_ROOT / 'informes' / 'fase3_modeling' / '03g_tuning_catboost'

best_params = {}
for target in TARGETS:
    with open(TUNING_DIR / f'best_params_{target}.json') as f:
        best_params[target] = json.load(f)
    print(f'best_params[{target}] = {best_params[target]}')

best_params[churn_14d] = {'iterations': 590, 'learning_rate': 0.04831657963496582, 'depth': 4, 'l2_leaf_reg': 5.641017040821281, 'border_count': 47, 'bagging_temperature': 0.24554232102189724, 'random_strength': 7.477893566373788, 'min_data_in_leaf': 6}
best_params[churn_30d] = {'iterations': 1218, 'learning_rate': 0.01644266940285884, 'depth': 7, 'l2_leaf_reg': 1.7408592532406044, 'border_count': 55, 'bagging_temperature': 0.005258618458008513, 'random_strength': 0.6822584674450662, 'min_data_in_leaf': 45}


In [2]:
# [EXEC] Cargar y preparar datos: combinar train+val (sin test)
master = pd.read_parquet(MASTER_PATH).reset_index(drop=True)
splits_df = pd.read_parquet(SPLITS_PATH).reset_index(drop=True)
if 'user_id' in splits_df.columns:
    splits_df = splits_df.set_index('user_id').reindex(master['user_id'].values).reset_index()

cat_cols = master.select_dtypes(include=['object', 'string', 'category']).columns.tolist()
cat_cols = [c for c in cat_cols if c != 'user_id']
for c in cat_cols:
    master[c] = master[c].astype(str).fillna('missing').replace('nan', 'missing')

trainval_mask = splits_df['split'].values != 'test'
X_full = master.drop(columns=['churn_14d', 'churn_30d', 'user_id'])
X_trainval = X_full[trainval_mask].reset_index(drop=True)
print(f'X_trainval shape: {X_trainval.shape}')

X_trainval shape: (21420, 77)


In [3]:
# [EXEC] CV k=5 estratificado por target
all_metrics = []
for target in TARGETS:
    print(f'\n=== CV {target} ===')
    y = master[target].astype(int)[trainval_mask].reset_index(drop=True)
    skf = StratifiedKFold(n_splits=K, shuffle=True, random_state=RANDOM_STATE)
    params = {**best_params[target], 'random_state': RANDOM_STATE,
              'eval_metric': 'AUC', 'verbose': False, 'thread_count': -1}
    for fold, (tr_idx, vl_idx) in enumerate(skf.split(X_trainval, y), start=1):
        t0 = time.time()
        X_tr, X_vl = X_trainval.iloc[tr_idx], X_trainval.iloc[vl_idx]
        y_tr, y_vl = y.iloc[tr_idx], y.iloc[vl_idx]
        train_pool = Pool(X_tr, y_tr, cat_features=cat_cols)
        val_pool = Pool(X_vl, y_vl, cat_features=cat_cols)
        model = CatBoostClassifier(**params)
        model.fit(train_pool, eval_set=val_pool, early_stopping_rounds=50, use_best_model=True)
        proba = model.predict_proba(val_pool)[:, 1]
        pred = (proba >= 0.5).astype(int)
        all_metrics.append({
            'target': target,
            'fold': fold,
            'n_train': len(tr_idx),
            'n_val': len(vl_idx),
            'auc_roc': float(roc_auc_score(y_vl, proba)),
            'auc_pr': float(average_precision_score(y_vl, proba)),
            'log_loss': float(log_loss(y_vl, proba)),
            'f1': float(f1_score(y_vl, pred)),
            'elapsed_s': time.time() - t0,
        })
        print(f'  fold {fold}: AUC={all_metrics[-1]["auc_roc"]:.4f} | t={all_metrics[-1]["elapsed_s"]:.1f}s')

cv_df = pd.DataFrame(all_metrics)
cv_df.to_csv(REPORT_DIR / 'cv_metrics.csv', index=False)
print('\n=== CV summary ===')
summary = cv_df.groupby('target').agg(
    auc_mean=('auc_roc', 'mean'),
    auc_std=('auc_roc', 'std'),
    auc_min=('auc_roc', 'min'),
    auc_max=('auc_roc', 'max'),
    pr_mean=('auc_pr', 'mean'),
    pr_std=('auc_pr', 'std'),
    f1_mean=('f1', 'mean'),
).round(4).reset_index()
print(summary.to_string(index=False))
summary.to_csv(REPORT_DIR / 'cv_summary.csv', index=False)


=== CV churn_14d ===


  fold 1: AUC=0.8589 | t=3.9s


  fold 2: AUC=0.8438 | t=3.8s


  fold 3: AUC=0.8592 | t=3.4s


  fold 4: AUC=0.8488 | t=3.8s


...

  fold 1: AUC=0.8100 | t=5.0s


  fold 2: AUC=0.8005 | t=8.6s


  fold 3: AUC=0.7976 | t=9.9s


  fold 4: AUC=0.7994 | t=11.4s


  fold 5: AUC=0.8013 | t=9.7s

=== CV summary ===
   target  auc_mean  auc_std  auc_min  auc_max  pr_mean  pr_std  f1_mean
churn_14d    0.8521   0.0067   0.8438   0.8592   0.8153  0.0079   0.7105
churn_30d    0.8017   0.0048   0.7976   0.8100   0.8206  0.0049   0.6805


In [4]:
# [REPORT] cv_summary.md
lines = [
    '# Cross-validation k=5 — CatBoost tuneado',
    '',
    f'**Fecha:** {datetime.now().strftime("%Y-%m-%d %H:%M")}',
    f'**Master:** master_table_cutoff_v3_aggressive.parquet',
    f'**Folds:** {K} estratificados sobre TRAIN+VAL ({trainval_mask.sum():,} usuarios)',
    f'**Test set:** intacto, no se evalúa aquí',
    '',
    '## Resultados',
    '',
    '| target | AUC mean | AUC std | AUC min | AUC max | AUC-PR mean | F1 mean |',
    '|---|---:|---:|---:|---:|---:|---:|',
]
for _, row in summary.iterrows():
    lines.append(
        f'| {row["target"]} | {row["auc_mean"]:.4f} | {row["auc_std"]:.4f} | {row["auc_min"]:.4f} | '
        f'{row["auc_max"]:.4f} | {row["pr_mean"]:.4f} | {row["f1_mean"]:.4f} |'
    )

lines += [
    '',
    '## Interpretación',
    '',
]
for _, row in summary.iterrows():
    if row['auc_std'] < 0.01:
        verdict = 'modelo robusto, baja variabilidad entre folds'
    elif row['auc_std'] < 0.02:
        verdict = 'estabilidad aceptable'
    else:
        verdict = 'variabilidad alta, posible inestabilidad'
    lines.append(f'- **{row["target"]}**: AUC {row["auc_mean"]:.4f} ± {row["auc_std"]:.4f} → {verdict}')

with open(REPORT_DIR / 'cv_summary.md', 'w') as f:
    f.write('\n'.join(lines))
print('\ncv_summary.md guardado')


cv_summary.md guardado
